In [1]:
# LOAD DATA
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter

# ── LOAD ──────────────────────────────────────────────
df = pd.read_csv("online_retail_II.csv", encoding='utf-8')
print(f"Raw rows: {len(df):,}")

# ── CLEAN ─────────────────────────────────────────────
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df.dropna(subset=['Customer ID', 'Description'])
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df = df[df['Country'] == 'United Kingdom']
df['Description'] = df['Description'].str.strip().str.upper()
print(f"Clean rows: {len(df):,}")

# ── BASKET MATRIX ─────────────────────────────────────
transactions = df.groupby('Invoice')['Description'].apply(set).reset_index()
transactions.columns = ['Invoice', 'Items']
transactions = transactions[transactions['Items'].apply(len) >= 2]
all_transactions = transactions['Items'].tolist()
n = len(all_transactions)
print(f"Baskets: {n:,}")

# ── APRIORI ───────────────────────────────────────────
MIN_SUP = 0.01
min_count = int(MIN_SUP * n)

item_counts = Counter(item for t in all_transactions for item in t)
freq1 = {frozenset([i]): c for i, c in item_counts.items() if c >= min_count}
freq_set = set(i for k in freq1 for i in k)

pair_counts = Counter()
for t in all_transactions:
    fi = sorted([i for i in t if i in freq_set])
    for pair in combinations(fi, 2):
        pair_counts[pair] += 1
freq2 = {frozenset(p): c for p, c in pair_counts.items() if c >= min_count}

triple_counts = Counter()
for t in all_transactions:
    fi = sorted([i for i in t if i in freq_set])
    if len(fi) >= 3:
        for triple in combinations(fi, 3):
            if all(frozenset(p) in freq2 for p in combinations(triple, 2)):
                triple_counts[triple] += 1
freq3 = {frozenset(t): c for t, c in triple_counts.items() if c >= min_count}

all_freq = {**freq1, **freq2, **freq3}
print(f"Frequent itemsets: {len(all_freq):,}")

# ── ASSOCIATION RULES ─────────────────────────────────
rules = []
for itemset, icount in {k: v for k, v in all_freq.items() if len(k) >= 2}.items():
    items = list(itemset)
    for size in range(1, len(items)):
        for ant in combinations(items, size):
            ant, con = frozenset(ant), itemset - frozenset(ant)
            ac, cc = all_freq.get(ant, 0), all_freq.get(con, 0)
            if not ac or not cc: continue
            sup = icount / n
            conf = icount / ac
            lift = conf / (cc / n)
            rules.append({
                'antecedents': ', '.join(sorted(ant)),
                'consequents': ', '.join(sorted(con)),
                'support': round(sup, 4),
                'confidence': round(conf, 4),
                'lift': round(lift, 4),
                'leverage': round(sup - (ac/n)*(cc/n), 6),
                'conviction': round(min((1 - cc/n) / (1 - conf), 999), 4) if conf < 1 else 999
            })

rules_df = pd.DataFrame(rules)
rules_df = rules_df[rules_df['lift'] > 1].sort_values('lift', ascending=False)
print(f"Rules generated: {len(rules_df):,}")
print(rules_df.head(10).to_string())

Raw rows: 1,067,371
Clean rows: 725,250
Baskets: 30,794
Frequent itemsets: 839
Rules generated: 538
                       antecedents                    consequents  support  confidence     lift  leverage  conviction
387   POPPY'S PLAYHOUSE LIVINGROOM      POPPY'S PLAYHOUSE BEDROOM   0.0106      0.8270  53.9529  0.010358      5.6908
386      POPPY'S PLAYHOUSE BEDROOM   POPPY'S PLAYHOUSE LIVINGROOM   0.0106      0.6886  53.9529  0.010358      3.1699
388      POPPY'S PLAYHOUSE KITCHEN   POPPY'S PLAYHOUSE LIVINGROOM   0.0114      0.6610  51.7948  0.011178      2.9124
389   POPPY'S PLAYHOUSE LIVINGROOM      POPPY'S PLAYHOUSE KITCHEN   0.0114      0.8931  51.7948  0.011178      9.1958
382      POPPY'S PLAYHOUSE BEDROOM      POPPY'S PLAYHOUSE KITCHEN   0.0131      0.8538  49.5148  0.012823      6.7226
383      POPPY'S PLAYHOUSE KITCHEN      POPPY'S PLAYHOUSE BEDROOM   0.0131      0.7589  49.5148  0.012823      4.0849
65   SET/6 RED SPOTTY PAPER PLATES    SET/6 RED SPOTTY PAPER CUPS   0.0110

In [2]:
# Item frequency
item_freq_df = pd.DataFrame([
    {'Item': i, 'Frequency': c, 'Support': round(c/n, 4)}
    for i, c in item_counts.items() if c >= min_count
]).sort_values('Frequency', ascending=False)
item_freq_df.to_csv("item_frequency.csv", index=False)

# Association rules
rules_df['antecedents'] = rules_df['antecedents'].astype(str)
rules_df['consequents'] = rules_df['consequents'].astype(str)
rules_df.to_csv("association_rules.csv", index=False)

# Heatmap matrix (top 20 items)
top20 = item_freq_df.head(20)['Item'].tolist()
rt = rules_df[rules_df['antecedents'].isin(top20) & rules_df['consequents'].isin(top20)]
heatmap_rows = [{'Antecedent': a, 'Consequent': c,
                 'Lift': rt[(rt['antecedents']==a)&(rt['consequents']==c)]['lift'].values[0]
                 if len(rt[(rt['antecedents']==a)&(rt['consequents']==c)]) > 0 else None,
                 'Confidence': rt[(rt['antecedents']==a)&(rt['consequents']==c)]['confidence'].values[0]
                 if len(rt[(rt['antecedents']==a)&(rt['consequents']==c)]) > 0 else None}
                for a in top20 for c in top20 if a != c]
pd.DataFrame(heatmap_rows).to_csv("heatmap_data.csv", index=False)

# Network edges (top 80 rules)
rules_df.nlargest(80,'lift')[['antecedents','consequents','lift','confidence','support']]\
        .rename(columns={'antecedents':'Source','consequents':'Target','lift':'Weight'})\
        .to_csv("network_edges.csv", index=False)

# KPI summary
pd.DataFrame([{
    'Total_Transactions': n,
    'Total_Rules': len(rules_df),
    'Max_Lift': round(rules_df['lift'].max(), 2),
    'Avg_Confidence': round(rules_df['confidence'].mean(), 4),
    'Avg_Lift': round(rules_df['lift'].mean(), 2),
    'Rules_Lift_Above_5': int((rules_df['lift'] > 5).sum())
}]).to_csv("kpi_summary.csv", index=False)

print("✅ All 5 CSVs saved! Ready for Tableau.")

✅ All 5 CSVs saved! Ready for Tableau.
